In [31]:
from typing import List, Set
import numpy as np
from networkx import Graph
from skimage.measure import approximate_polygon

epsilon = 1
tolerance = 0.2

def simplify_network(
    net: Graph, 
    epsilon: float = 2.0, 
    method: str = 'rdp'
) -> List[np.ndarray]:
    """
    Simplify the given network using the specified method.

    Args:
        net (gng.GNG): The input Growing Neural Gas network to simplify.
        epsilon (float, optional): The epsilon parameter for the simplification algorithm. Defaults to 2.0.
        method (str, optional): The simplification method to use ('rdp', 'edge_contraction', 'path_compression'). Defaults to 'rdp'.

    Returns:
        List[np.ndarray]: A list of simplified segments represented as numpy arrays.
    """
    if method == 'rdp':
        return simplify_with_rdp(net, epsilon)
    elif method == 'edge_contraction':
        return simplify_with_edge_contraction(net, epsilon)
    elif method == 'path_compression':
        return simplify_with_path_compression(net, epsilon)
    else:
        raise ValueError(f"Unsupported simplification method: {method}")

def simplify_with_rdp(net, epsilon: float) -> List[np.ndarray]:
    """
    Simplify the network using skimage's polygon approximation algorithm while preserving closed contours.

    Args:
        net: The input Growing Neural Gas network to simplify.
        tolerance (float): The maximum distance from original points of polygon to approximated polygonal chain.

    Returns:
        List[np.ndarray]: A list of simplified segments.
    """
    simplified_segments: List[np.ndarray] = []
    visited_nodes: Set[int] = set()

    def traverse_segment(start: int, is_closed: bool = False) -> np.ndarray:
        segment = [net.w[start]]
        current = start
        first = start
        while True:
            visited_nodes.add(current)
            neighbors = set(np.where(net.C[current] == 1)[0]) - visited_nodes
            if not neighbors:
                if is_closed and current != first:
                    # Complete the loop for closed contours
                    segment.append(net.w[first])
                break
            next_node = neighbors.pop()
            segment.append(net.w[next_node])
            if next_node == first:
                break
            if np.sum(net.C[next_node]) != 2 and not is_closed:
                break
            current = next_node
        return np.array(segment)

    def simplify_contour(contour: np.ndarray, is_closed: bool) -> np.ndarray:
        # Approximate the polygon
        simplified = approximate_polygon(contour, tolerance=tolerance)
        
        # Ensure the contour is closed if it's supposed to be
        if is_closed and not np.array_equal(simplified[0], simplified[-1]):
            simplified = np.vstack([simplified, simplified[0]])
        
        return simplified

    def process_node(node: int) -> None:
        if node in visited_nodes:
            return
        neighbors = set(np.where(net.C[node] == 1)[0])
        if len(neighbors) == 2:  # This might be part of a closed contour
            segment = traverse_segment(node, is_closed=True)
            if len(segment) > 3:  # Minimum points for a meaningful closed shape
                simplified_segment = simplify_contour(segment, is_closed=True)
                simplified_segments.append(simplified_segment)
        else:
            for neighbor in neighbors - visited_nodes:
                segment = traverse_segment(node)
                if len(segment) > 2:
                    simplified_segment = simplify_contour(segment, is_closed=False)
                    simplified_segments.append(simplified_segment)

    # Process all nodes
    for node in range(len(net.w)):
        process_node(node)

    return simplified_segments

def simplify_with_edge_contraction(net: Graph, epsilon: float) -> List[np.ndarray]:
    """
    Simplify the network using the Edge Contraction algorithm.

    Args:
        net (nx.Graph): The input graph to simplify.
        epsilon (float): A threshold parameter for edge contraction.

    Returns:
        List[np.ndarray]: A list of simplified segments.
    """
    simplified_net = net.copy()
    
    def contract_edge(u: int, v: int) -> None:
        # Calculate the midpoint of the edge
        mid_x = (simplified_net.nodes[u]['x'] + simplified_net.nodes[v]['x']) / 2
        mid_y = (simplified_net.nodes[u]['y'] + simplified_net.nodes[v]['y']) / 2
        
        # Create a new node at the midpoint
        new_node = max(simplified_net.nodes()) + 1
        simplified_net.add_node(new_node, x=mid_x, y=mid_y)
        
        # Connect the new node to all neighbors of u and v
        for neighbor in set(simplified_net.neighbors(u)) | set(simplified_net.neighbors(v)):
            if neighbor not in (u, v):
                simplified_net.add_edge(new_node, neighbor)
        
        # Remove the original nodes
        simplified_net.remove_node(u)
        simplified_net.remove_node(v)

    # Iterate through edges and contract those shorter than epsilon
    edges_to_contract = [(u, v) for u, v in simplified_net.edges() 
                         if np.linalg.norm(np.array([simplified_net.nodes[u]['x'], simplified_net.nodes[u]['y']]) - 
                                           np.array([simplified_net.nodes[v]['x'], simplified_net.nodes[v]['y']])) < epsilon]
    
    for u, v in edges_to_contract:
        if u in simplified_net.nodes() and v in simplified_net.nodes():
            contract_edge(u, v)

    # Convert the simplified graph back to segments
    simplified_segments = []
    visited_edges = set()

    for u in simplified_net.nodes():
        for v in simplified_net.neighbors(u):
            if (u, v) not in visited_edges and (v, u) not in visited_edges:
                visited_edges.add((u, v))
                simplified_segments.append(np.array([[simplified_net.nodes[u]['x'], simplified_net.nodes[u]['y']],
                                                     [simplified_net.nodes[v]['x'], simplified_net.nodes[v]['y']]]))

    return simplified_segments

def simplify_with_path_compression(net: Graph, epsilon: float) -> List[np.ndarray]:
    """
    Simplify the network using the Path Compression algorithm.

    Args:
        net (gng.GNG): The input Growing Neural Gas network to simplify.
        epsilon (float): A threshold parameter for path compression.

    Returns:
        List[np.ndarray]: A list of simplified segments.
    """
    degree = np.sum(net.C, axis=0)
    endpoints = set(np.where(degree == 1)[0])
    intersections = set(np.where(degree > 2)[0])

    simplified_segments: List[np.ndarray] = []
    visited_nodes: Set[int] = set()

    def traverse_and_compress(start: int) -> List[np.ndarray]:
        segment = [net.w[start]]
        current = start
        prev = -1
        while True:
            visited_nodes.add(current)
            neighbors = set(np.where(net.C[current] == 1)[0]) - {prev}
            if not neighbors:
                break
            if len(neighbors) > 1:
                break  # Intersection node
            next_node = neighbors.pop()
            segment.append(net.w[next_node])
            if next_node in endpoints or next_node in intersections:
                break
            prev, current = current, next_node
        # Apply compression if segment length exceeds epsilon
        if len(segment) > epsilon:
            simplified = segment[::int(len(segment)/epsilon)]
            return simplified
        return segment

    def process_node(node: int) -> None:
        if node in visited_nodes:
            return
        neighbors = set(np.where(net.C[node] == 1)[0]) - visited_nodes
        for neighbor in neighbors:
            segment = traverse_and_compress(node)
            if len(segment) > 2:
                simplified_segments.append(np.array(segment))

    # Process all endpoints and intersections
    for node in endpoints.union(intersections):
        process_node(node)

    # Handle closed contours (no endpoints)
    unvisited_nodes = set(range(len(net.w))) - visited_nodes
    while unvisited_nodes:
        start_node = unvisited_nodes.pop()
        segment = traverse_and_compress(start_node)
        if len(segment) > 2:
            simplified_segments.append(np.array(segment))
        unvisited_nodes -= visited_nodes

    return simplified_segments

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ypstruct import structure
import gng
import cv2
import os
from skimage.morphology import skeletonize
from skimage.util import img_as_ubyte
from rdp import rdp
from typing import List, Tuple
epsilon = 1

# Read all images from the specified folder
folder_path = '../../tests/generated_samples/mnist_fours'  # Replace with the actual folder path
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif'))]

for image_file in image_files:
    # Read the image
    data = cv2.imread(os.path.join(folder_path, image_file), 0)

    # Skeletonize the image
    binary = data > 140
    skeleton = skeletonize(binary)
    skeleton = img_as_ubyte(skeleton)

    h, w = skeleton.shape

    points = []

    for y in range(h):
        for x in range(w):
            # Threshold the pixel
            if skeleton[y, x] > 0:
                points.append([x, y])

    points_np = np.array(points)


    # Neural Gas Parameters
    params = structure()
    params.N = 35
    params.maxit = 100
    params.L = 100
    params.epsilon_b = 0.2
    params.epsilon_n = 0.01
    params.alpha = 0.5
    params.delta = 0.995
    params.T = 50

    # Fit Neural Gas to Data
    print(f"Fitting Growing Neural Gas Network for {image_file}...")
    net = gng.fit(points_np, params)
    
    # Simplify the network
    simplified_network = simplify_network(net, epsilon=epsilon, method='rdp')

    # Create a figure with four subplots side by side
    fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1, 5, figsize=(40, 10))

    # Plot original image on the left
    ax1.imshow(data, cmap='gray')
    ax1.set_title(f'Original Image: {image_file}')
    ax1.axis('off')
    
    ax2.imshow(binary, cmap='gray')
    ax2.set_title(f'Binary Image: {image_file}')
    ax2.axis('off')

    # Plot skeletonized image in the middle-left
    ax3.imshow(skeleton, cmap='gray')
    ax3.set_title(f'Skeletonized Image: {image_file}')
    ax3.axis('off')

    # Plot GNG result in the middle-right
    ax4.grid()
    ax4.scatter(points_np[:, 0], points_np[:, 1], s=2)
    for i in range(params.N):
        for j in range(i+1, params.N):
            if net.C[i, j] == 1:
                ax4.plot([net.w[i, 0], net.w[j, 0]], [net.w[i, 1], net.w[j, 1]], c='r')

    ax4.scatter(net.w[:, 0], net.w[:, 1], s=60, c='y', edgecolors='r')

    ax4.set_title(f'GNG for {image_file}')
    ax4.set_xlabel('x')
    ax4.set_ylabel('y')
    ax4.axis('equal')
    ax4.invert_yaxis()  # Invert y-axis to match image coordinates

    # Plot simplified network on the right
    ax5.grid()
    ax5.scatter(points_np[:, 0], points_np[:, 1], s=2, alpha=0.3)
    for segment in simplified_network:
        ax5.plot(segment[:, 0], segment[:, 1], 'r-', linewidth=2)

    ax5.set_title(f'Simplified Network for {image_file}')
    ax5.set_xlabel('x')
    ax5.set_ylabel('y')
    ax5.axis('equal')
    ax5.invert_yaxis()  # Invert y-axis to match image coordinates

    plt.tight_layout()
    plt.show()

In [ ]:
from skeleton_gng_mapper import SkeletonGNGMapper
from graph_serializer import GraphSerializer
import networkx as nx
from settings import Settings
import matplotlib.pyplot as plt
import cv2
import json

# path = '../../tests/generated_samples/sk-test/mnist_one_00000.png'
path = '../../tests/generated_samples/sk-test/mnist_four_00003.png'
image = cv2.imread(path, 0)
settings = Settings(
    kafka_topic='skeletonization',
    dlq_topic='skeletonization-dlq',
    kafka_bootstrap_servers='localhost:9092',
    simplification_epsilon=2.5
)

# Process the image
graph = SkeletonGNGMapper(settings).process_image(image)

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Plot the original image
ax1.imshow(image, cmap='gray')
ax1.set_title('Original Image')
ax1.axis('off')

# Create a position dictionary using the node coordinates
pos = {node: (data['x'], data['y']) for node, data in graph.nodes(data=True)}

# Plot the graph
nx.draw(graph, pos, ax=ax2, with_labels=False, node_size=50, node_color='skyblue', edge_color='gray', alpha=0.7)

ax2.set_title('Growing Neural Gas Network')
ax2.axis('equal')  # Set equal aspect ratio
ax2.invert_yaxis()  # Invert y-axis to match image coordinates

plt.tight_layout()
plt.show()

json.loads(json.dumps(GraphSerializer.serialize(graph)))